In [0]:
# Databricks notebook source
from pyspark.sql.functions import col, trim, to_date, round as _round

df_claims_bronze = spark.read.table("workspace.bronze.claims")

# Dynamically resolve cost column name with potential leading whitespace
cost_col = next((c for c in df_claims_bronze.columns if c.strip() == "TOTAL"), "TOTAL")
id_col = "id" if "id" in df_claims_bronze.columns else "ID"

df_claims_silver = (
    df_claims_bronze
    .filter(col(id_col).isNotNull())
    .dropDuplicates([id_col])
    .select(
        col(id_col).alias("claim_id"),
        col("patient").alias("patient_id"),
        to_date(col("billableperiod"), "yyyy-MM-dd").alias("billable_period"),
        trim(col("organization")).alias("organization"),
        col("encounter").alias("encounter_id"),
        col("diagnosis").alias("diagnosis_code"),
        _round(col(cost_col).cast("double"), 2).alias("total_cost"),
        col("ingested_at")
    )
)

# Data Quality check for negative costs
assert df_claims_silver.filter(col("total_cost") < 0).count() == 0, "❌ DQ Error: Negative claim cost found!"

(
    df_claims_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.claims")
)

print(f"✅ Created workspace.silver.claims with {df_claims_silver.count()} clean rows!")